Code Khai báo và Tiền xử lý (Preprocessing)

In [3]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score

# 1. ĐỌC DỮ LIỆU
# Lưu ý: Thay đổi đường dẫn nếu cần
df_review = pd.read_csv('../data/processed/CAR.csv')

# 2. TIỀN XỬ LÝ (PREPROCESSING)
print("--- ĐANG TIỀN XỬ LÝ DỮ LIỆU ---")
# Drop các dòng thiếu Rating (biến mục tiêu)
df_review = df_review.dropna(subset=['Rating'])

# Điền giá trị khuyết (Missing Values) bằng Mode cho các biến phân loại
df_review['Class'] = df_review['Class'].fillna(df_review['Class'].mode()[0])
df_review['Traveller_type'] = df_review['Traveller_type'].fillna(df_review['Traveller_type'].mode()[0])

# Loại bỏ các cột văn bản không dùng cho mô hình toán học
cols_to_drop = ['Passanger_Name', 'Review_title', 'Review_content', 'Aircraft', 'Flying_month', 'Route']
df_ml = df_review.drop(columns=cols_to_drop)

# Tạo biến nhị phân cho bài toán Phân loại (Logistic & Decision Tree)
# Giả sử Rating >= 6 là Hài lòng (1), ngược lại là Không hài lòng (0)
df_ml['Is_Satisfied'] = (df_ml['Rating'] >= 6).astype(int)

print(f"Kích thước dữ liệu sau xử lý: {df_ml.shape}\n")

--- ĐANG TIỀN XỬ LÝ DỮ LIỆU ---
Kích thước dữ liệu sau xử lý: (3575, 5)



Code Phân tích Thống kê (Statistical Tests)

In [4]:
print("=============================================")
print("1. KIỂM ĐỊNH ANOVA (Phân tích phương sai)")
print("=============================================")
# Câu hỏi: Điểm Rating trung bình có khác biệt giữa các Hạng ghế (Class) không?
classes = df_ml['Class'].unique()
data_groups = [df_ml[df_ml['Class'] == cls]['Rating'] for cls in classes]

f_stat, p_val_anova = stats.f_oneway(*data_groups)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_val_anova:.4e}")
if p_val_anova < 0.05:
    print("-> Kết luận: Có sự khác biệt có ý nghĩa thống kê về điểm Rating giữa các hạng ghế.\n")
else:
    print("-> Kết luận: Không có sự khác biệt rõ rệt về điểm Rating giữa các hạng ghế.\n")

print("=============================================")
print("2. KIỂM ĐỊNH CHI-SQUARE (Chi bình phương)")
print("=============================================")
# Câu hỏi: Có mối liên hệ nào giữa Loại khách (Traveller_type) và Trạng thái xác thực (Verified) không?
contingency_table = pd.crosstab(df_ml['Traveller_type'], df_ml['Verified'])
chi2, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_table)

print(f"Chi2 value: {chi2:.4f}")
print(f"P-value: {p_val_chi2:.4e}")
if p_val_chi2 < 0.05:
    print("-> Kết luận: Có mối liên hệ phụ thuộc giữa Loại hành khách và Trạng thái xác thực.\n")
else:
    print("-> Kết luận: Hai biến này hoàn toàn độc lập với nhau.\n")

1. KIỂM ĐỊNH ANOVA (Phân tích phương sai)
F-statistic: 34.0192
P-value: 2.3152e-15
-> Kết luận: Có sự khác biệt có ý nghĩa thống kê về điểm Rating giữa các hạng ghế.

2. KIỂM ĐỊNH CHI-SQUARE (Chi bình phương)
Chi2 value: 496.0834
P-value: 5.8665e-104
-> Kết luận: Có mối liên hệ phụ thuộc giữa Loại hành khách và Trạng thái xác thực.



Code Học máy (Machine Learning Models)

In [5]:
# --- CHUẨN BỊ DỮ LIỆU CHO HỌC MÁY ---
# Mã hóa One-Hot cho các biến phân loại
df_encoded = pd.get_dummies(df_ml, columns=['Class', 'Traveller_type', 'Verified'], drop_first=True)

# Chia tập Train/Test cho bài toán Hồi quy (Dự đoán Rating)
X_reg = df_encoded.drop(columns=['Rating', 'Is_Satisfied'])
y_reg = df_encoded['Rating']
X_train_R, X_test_R, y_train_R, y_test_R = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Chia tập Train/Test cho bài toán Phân loại (Dự đoán Is_Satisfied)
X_clf = df_encoded.drop(columns=['Rating', 'Is_Satisfied'])
y_clf = df_encoded['Is_Satisfied']
X_train_C, X_test_C, y_train_C, y_test_C = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)


print("=============================================")
print("3. MÔ HÌNH HỒI QUY TUYẾN TÍNH (Linear Regression)")
print("=============================================")
lin_model = LinearRegression()
lin_model.fit(X_train_R, y_train_R)
y_pred_R = lin_model.predict(X_test_R)

print(f"R-squared (Độ phù hợp): {r2_score(y_test_R, y_pred_R):.4f}")
print(f"MSE (Sai số toàn phương trung bình): {mean_squared_error(y_test_R, y_pred_R):.4f}")
print("\nTop 3 Trọng số ảnh hưởng nhất:")
coef_df = pd.DataFrame({'Đặc trưng': X_reg.columns, 'Trọng số': lin_model.coef_}).sort_values(by='Trọng số', ascending=False)
print(coef_df.head(3).to_string(index=False), "\n")


print("=============================================")
print("4. MÔ HÌNH HỒI QUY LOGISTIC (Logistic Regression)")
print("=============================================")
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_C, y_train_C)
y_pred_log = log_model.predict(X_test_C)

print(f"Độ chính xác (Accuracy): {accuracy_score(y_test_C, y_pred_log):.4f}")
print("Báo cáo phân loại:\n", classification_report(y_test_C, y_pred_log, zero_division=0))


print("=============================================")
print("5. MÔ HÌNH CÂY QUYẾT ĐỊNH (Decision Tree Classifier)")
print("=============================================")
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train_C, y_train_C)
y_pred_dt = dt_model.predict(X_test_C)

print(f"Độ chính xác (Accuracy): {accuracy_score(y_test_C, y_pred_dt):.4f}")
print("Báo cáo phân loại:\n", classification_report(y_test_C, y_pred_dt, zero_division=0))

3. MÔ HÌNH HỒI QUY TUYẾN TÍNH (Linear Regression)
R-squared (Độ phù hợp): 0.1071
MSE (Sai số toàn phương trung bình): 9.3161

Top 3 Trọng số ảnh hưởng nhất:
                    Đặc trưng  Trọng số
  Traveller_type_Solo Leisure  1.669880
             Verified_Unknown  1.615027
Traveller_type_Couple Leisure  1.259716 

4. MÔ HÌNH HỒI QUY LOGISTIC (Logistic Regression)
Độ chính xác (Accuracy): 0.6252
Báo cáo phân loại:
               precision    recall  f1-score   support

           0       0.64      0.86      0.73       430
           1       0.56      0.27      0.37       285

    accuracy                           0.63       715
   macro avg       0.60      0.57      0.55       715
weighted avg       0.61      0.63      0.59       715

5. MÔ HÌNH CÂY QUYẾT ĐỊNH (Decision Tree Classifier)
Độ chính xác (Accuracy): 0.6462
Báo cáo phân loại:
               precision    recall  f1-score   support

           0       0.66      0.85      0.74       430
           1       0.60      0.34     

BIỂU ĐỒ KIỂM ĐỊNH HẬU ĐỊNH TUKEY HSD

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ==========================================
# BƯỚC 1: ĐỌC DỮ LIỆU
# ==========================================
df = pd.read_csv('../data/processed/CAR.csv') 

# THÊM 3 DÒNG NÀY VÀO ĐỂ KIỂM TRA:
print("Tổng số cột:", len(df.columns))
print("Danh sách các cột thực tế trong file này là:")
print(df.columns.tolist())

# Fix nhanh lỗi khoảng trắng (nếu có):
df.columns = df.columns.str.strip() 

# (Sau khi chạy xong, bạn nhìn xuống kết quả print xem có cột purchase_lead không)

Tổng số cột: 10
Danh sách các cột thực tế trong file này là:
['Passanger_Name', 'Flying_month', 'Route', 'Rating', 'Verified', 'Review_title', 'Review_content', 'Traveller_type', 'Class', 'Aircraft']
